<a href="https://colab.research.google.com/github/winter-pro/Statistical-Learning-e23091/blob/main/Assignment_7b_Gaussian_Mixture_Model_Clustering_as_Conditional_Updating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

Task 1: Visualizing the Mechanics

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL IRT response probability function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-4, 4, 400)

# Scenarios to plot
# 1. Base case: a=1.0, b=0.0
# 2. Moving b (horizontal shifts): a=1.0, b=-1.5 and a=1.0, b=1.5
# 3. High discrimination: a=2.5, b=0.0
fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_vals, y=p_i(theta_vals, 1.0, 0.0), name="a=1.0, b=0.0 (Baseline)"))
fig.add_trace(go.Scatter(x=theta_vals, y=p_i(theta_vals, 1.0, -1.5), name="a=1.0, b=-1.5 (Easy Item)"))
fig.add_trace(go.Scatter(x=theta_vals, y=p_i(theta_vals, 1.0, 1.5), name="a=1.0, b=1.5 (Hard Item)"))
fig.add_trace(go.Scatter(x=theta_vals, y=p_i(theta_vals, 2.5, 0.0), name="a=2.5, b=0.0 (High Discrim.)"))

fig.update_layout(
    title="2PL IRT Item Characteristic Curves (ICCs)",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i=1|θ)",
    template="plotly_white"
)
fig.show()

Interpretation: Moving the difficulty parameter $b_i$ shifts the curve horizontally. A larger $b_i$ moves the curve to the right, meaning a user needs a higher latent ability $\theta$ to achieve a 50% probability of a correct response. Conversely, a lower $b_i$ shifts it left, indicating a simpler item where lower ability yields success.

Task 2: Sequential Likelihood Contribution

The likelihood contribution $L(y_k \mid \theta)$ of a single isolated response $y_k$ at step $k$ is given by:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} = \left(\frac{1}{1+e^{-a_k(\theta-b_k)}}\right)^{y_k} \left(\frac{e^{-a_k(\theta-b_k)}}{1+e^{-a_k(\theta-b_k)}}\right)^{1-y_k}$$Assuming local independence conditional on $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of individual step likelihoods:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

Task 3: Mathematical Formulation of the Running Update

Using Bayes' Theorem recursively, the running posterior distribution at step $k$ is proportional to the product of the single-step likelihood contribution and the prior state (which is the posterior from step $k-1$):$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left( \frac{1}{1+e^{-a_k(\theta-b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta-b_k)}}{1+e^{-a_k(\theta-b_k)}} \right)^{1-y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Task 4: Dynamic Shifting

When a user correctly answers a highly difficult item ($y_k = 1$, large $b_k$), the likelihood contribution $L(1 \mid \theta) = p_k(\theta)$ acts as a monotonically increasing multiplier that is near zero for low $\theta$ and climbs sharply near $\theta = b_k$. Multiplying the prior $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ by this increasing function suppresses the lower tail of the distribution and accentuates the upper tail, mathematically shifting the mode (peak) of the running posterior density distinctly to the right.

Task 5: Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the steepness of the item response function. When $a_k$ is very large, the likelihood function resembles a sharp step-function, which causes a substantial, narrow modification to the posterior, drastically reducing its variance and increasing its "sharpness" (certainty). When $a_k$ is very small, the curve is flat and uninformative, adding very little new information, meaning the posterior variance barely shifts.

Task 6: Numerical Implementation of a Running Grid

Define a fixed grid of points over the domain of $\theta$ (e.g., $N=1000$ points evenly spaced from $[-4, 4]$).Initialize the grid with evaluations of the standard normal prior PDF $f_{\Theta}^{(0)}(\theta)$. Normalize via standard rectangular/trapezoidal scaling so the sum equals $1$.When new data $y_k$ arrives for an item with parameters $(a_k, b_k)$, calculate the likelihood vector $L(y_k \mid \theta)$ across the entire grid vector.Perform an element-wise multiplication: $\text{unnormalized\_posterior} = L(y_k \mid \theta) \odot \text{prior}$.Perform the sequential normalization computationally:$$\text{normalized\_posterior} = \frac{\text{unnormalized\_posterior}}{\sum_{j=1}^N \text{unnormalized\_posterior}_j \cdot \Delta \theta}$$

Task 7: Evaluating Convergence over the Timeline

In [2]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n_items = 20
theta_true = 0.75
theta_grid = np.linspace(-4, 4, 1000)
d_theta = theta_grid[1] - theta_grid[0]

# Generate item parameters
b_items = np.random.normal(0, 1, n_items)
a_items = np.random.uniform(0.5, 2.0, n_items)

# Initialize Prior
posterior = (1 / np.sqrt(2 * np.pi)) * np.exp(-theta_grid**2 / 2)
posterior /= np.sum(posterior * d_theta)

bayes_estimates = [0.0]  # Prior mean
map_estimates = [0.0]    # Prior mode

for k in range(n_items):
    a_k = a_items[k]
    b_k = b_items[k]

    # Simulate Response
    p_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Calculate Likelihood
    p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = p_grid if y_k == 1 else (1 - p_grid)

    # Sequential update & Normalize
    posterior = posterior * likelihood
    posterior /= np.sum(posterior * d_theta)

    # Compute point estimators
    hat_theta_bayes = np.sum(theta_grid * posterior * d_theta)
    hat_theta_map = theta_grid[np.argmax(posterior)]

    bayes_estimates.append(hat_theta_bayes)
    map_estimates.append(hat_theta_map)

# Visualization
steps = list(range(n_items + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines+markers', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), mode='lines', name='True Theta (0.75)', line=dict(dash='dash', color='black')))

fig.update_layout(title="Convergence of Bayesian Estimators Over Items", xaxis_title="Item Number (k)", yaxis_title="Estimate", template="plotly_white")
fig.show()

Analysis: As $k$ increases, the distance between the estimators and $\theta_{\text{true}}$ generally decreases, converging toward the true value. This behavior implies that the platform's confidence in its measurement expands with every item delivered, as the accumulation of evidence continuously compresses the posterior variance.

Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

Task 1: Structural Probability and Properties

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_vals = np.linspace(0, 1, 500)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_vals, y=stats.beta.pdf(theta_vals, 1, 1), name="α=1, β=1 (Uniform)"))
fig.add_trace(go.Scatter(x=theta_vals, y=stats.beta.pdf(theta_vals, 2, 8), name="α=2, β=8 (Right-skewed)"))
fig.add_trace(go.Scatter(x=theta_vals, y=stats.beta.pdf(theta_vals, 8, 2), name="α=8, β=2 (Left-skewed)"))

fig.update_layout(title="Beta Distribution PDF Shapes", xaxis_title="θ", yaxis_title="Density", template="plotly_white")
fig.show()

Interpretation: The balance between $\alpha$ and $\beta$ determines the center of mass. When $\alpha = \beta$, the distribution is symmetric. When $\alpha < \beta$, the density shifts to the left (skewed right), indicating a prior belief weighted toward lower values. When $\alpha > \beta$, it moves toward the right (skewed left), signifying an expectation of higher values.

Task 2: Sequential Likelihood and Joint History

The mathematical likelihood contribution of a single observation $y_k$ is:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$Assuming conditionally independent trials, the joint likelihood function for $\mathbf{y}^{(k)}$ is:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum y_i} (1 - \theta)^{k - \sum y_i}$$

Task 3: Closed-Form Analytical Updates (Conjugacy)

By Bayes' Theorem:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the structural functional forms:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[\theta^{y_k} (1 - \theta)^{1 - y_k}\right] \cdot \left[\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}\right]$$$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$This matches the functional core of a Beta distribution, proving algebraic Beta-Binomial Conjugacy. The closed-form recursive update parameters are:$$\alpha_k = \alpha_{k-1} + y_k, \qquad \beta_k = \beta_{k-1} + (1 - y_k)$$The closed-form analytical expression for the Posterior Mean at step $k$ is:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

Task 4: Dynamic Shifting Mechanics

An observed click ($y_k = 1$) increments $\alpha_k$, pulling the distribution's weight toward 1. A non-click ($y_k = 0$) increments $\beta_k$, dragging the peak toward 0. In this conjugate setup, updating requires only simple additions to the parameters. In contrast, non-conjugate setups (like the 2PL IRT model) cannot be factored cleanly into common distribution families, strictly requiring expensive numerical grid evaluation or sampling frameworks at every step.

Task 5: Running Point Estimators

Running Posterior Mean: $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$Running Maximum A Posteriori (MAP): $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ (for $\alpha_k > 1, \beta_k > 1$)

Task 6: Performance Tracking and Convergence Analysis

In [4]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n_impressions = 100
theta_true = 0.35

alpha, beta = 1, 1  # Initialization
bayes_tracking = [alpha / (alpha + beta)]
map_tracking = [0.5] # Midpoint for uniform initial map

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha += y_k
    beta += (1 - y_k)

    bayes_tracking.append(alpha / (alpha + beta))
    map_tracking.append((alpha - 1) / (alpha + beta - 2) if (alpha + beta > 2) else 0.5)

steps = list(range(n_impressions + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=bayes_tracking, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=map_tracking, mode='lines', name='MAP Estimate'))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), mode='lines', name='True CTR', line=dict(dash='dash', color='black')))

fig.update_layout(title="Beta-Binomial Sequential Update Convergence", xaxis_title="Impressions (k)", yaxis_title="Estimated CTR", template="plotly_white")
fig.show()

Analysis: As the sampling size $k$ approaches 100, the distance between the estimators and $\theta_{\text{true}}$ narrows significantly. This indicates that as empirical evidence accumulates, the likelihood dominates the calculation, rendering the specific choice of the initial prior less influential on the final parameter estimation.

Q3. Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

Task 1: Prior Belief Boundaries

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0.01, 1.0, 500)
prior_pdf = stats.beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure(go.Scatter(x=theta_grid, y=prior_pdf, name="Prior Beta(8, 1.5)"))
fig.update_layout(title="Initial Structural Health Prior Density", xaxis_title="Stiffness Factor (θ)", yaxis_title="Density", template="plotly_white")
fig.show()

The expected prior value is computed as:$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$This specific distribution serves as an ideal structural prior because its density clusters strongly towards 1.0, matching the physical engineering assumption that a deployed component begins its operational lifecycle in a pristine, healthy condition.

Task 2: Structural Likelihood Formulation

Given $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$, where $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the term $\log\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right) = \epsilon_k$ follows a normal distribution. By change of variables, the single-measurement likelihood contribution is:$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\log(y_k) - \log(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$Assuming independence, the joint likelihood for history vector $\mathbf{y}^{(k)}$ is:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\log(y_i) - \log(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

An exact closed-form analytical solution does not exist because multiplying a Beta prior polynomial $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ by a Log-Normal likelihood containing $\log(\theta)$ inside an exponent cannot be simplified into any standard probability distribution.The recursive relationship is:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \exp\left( -\frac{\left(\log(y_k) - \log(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Task 4: Running Point Estimates

Running Posterior Mean: $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}{\int_{0}^{1} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}$Running Maximum A Posteriori (MAP): $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$

Task 5: Algorithmic Grid Approximation and Normalization

Discretize the domain into a uniform array (e.g., $N=1000$ steps from $0.01$ to $1.0$).Evaluate the prior density at each grid point to form the initial unnormalized array.Upon receiving measurement $y_k$, compute the likelihood vector across the grid array using the Log-Normal formulation.Update the grid point-by-point via element-wise multiplication: $\text{unnormalized\_post} = \text{prior\_grid} \odot \text{likelihood\_grid}$.Normalize the distribution utilizing the composite trapezoidal rule via np.trapezoid or manually:$$I = \frac{\Delta \theta}{2} \left[ g(\theta_1) + 2\sum_{j=2}^{N-1} g(\theta_j) + g(\theta_N) \right], \qquad \text{normalized\_grid} = \frac{\text{unnormalized\_post}}{I}$$

Task 6: Performance Tracking and Degradation Convergence Analysis

In [7]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)
n_steps = 15
K_nominal = 50.0
sigma = 0.15
theta_true = 0.68

theta_grid = np.linspace(0.01, 1.0, 1000)
d_theta = theta_grid[1] - theta_grid[0]

# Initialize Prior
posterior = stats.beta.pdf(theta_grid, 8, 1.5)
posterior /= np.trapezoid(posterior, theta_grid)

saved_posteriors = {0: posterior.copy()}
bayes_est = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_est = [theta_grid[np.argmax(posterior)]]

milestones = [1, 2, 5, 10, 15]

for k in range(1, n_steps + 1):
    # Physics generation
    epsilon = np.random.normal(0, sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon)

    # Likelihood updates
    log_diff = log_diff = np.log(y_k) - np.log(theta_grid * K_nominal)
    likelihood = np.exp(- (log_diff)**2 / (2 * sigma**2))

    posterior *= likelihood
    posterior /= np.trapezoid(posterior, theta_grid)

    if k in milestones:
        saved_posteriors[k] = posterior.copy()

    bayes_est.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_est.append(theta_grid[np.argmax(posterior)])

# Plot 1: Densities
fig1 = go.Figure()
for k, post in saved_posteriors.items():
    fig1.add_trace(go.Scatter(x=theta_grid, y=post, name=f"Step k={k}"))
fig1.update_layout(title="Evolution of Posterior Stiffness Density Curves", xaxis_title="θ", yaxis_title="Density", template="plotly_white")
fig1.show()

# Plot 2: Timeline convergence
steps = list(range(n_steps + 1))
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=bayes_est, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=steps, y=map_est, mode='lines+markers', name='MAP Estimate'))
fig2.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), mode='lines', name='True Stiffness (0.68)', line=dict(dash='dash', color='black')))
fig2.add_trace(go.Scatter(x=[0], y=[0.842], mode='markers', name='Prior Expected Value', marker=dict(color='red', size=10)))
fig2.update_layout(title="Stiffness Estimation Timeline Convergence", xaxis_title="Inspection Step (k)", yaxis_title="Stiffness Factor (θ)", template="plotly_white")
fig2.show()

Analysis: The model shifts away from its initial optimistic prior and isolates the 68% damage state within approximately 2 to 5 sensor readings. The rapid narrowing of the density curves reveals dropping variance, which yields precise structural tracking. This narrowing implies that engineers can set tight safety thresholds; a narrow distribution that crosses beneath an acceptable structural threshold triggers highly reliable, early maintenance warnings.

Q4. Gaussian Mixture Clustering as Conditional Updating

Task 1: Deriving the Marginal Density

By the Law of Total Probability, the marginal density $p(x_i)$ is found by summing the joint probability over all possible mutually exclusive discrete states of the latent variable $C_i$:$$p(x_i) = \sum_{k=1}^K p(x_i, C_i=k) = \sum_{k=1}^K P(C_i=k)p(x_i \mid C_i=k)$$Substituting $P(C_i=k) = \phi_k$ and $p(x_i \mid C_i=k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ yields:$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$This is called a Gaussian mixture density because it is a convex combination (a blend weighted by probabilities $\phi_k$) of $K$ distinct multivariate normal component densities.

Task 2: Deriving the Posterior Cluster Probability

Applying Bayes' rule for a specific observation $x_i$ gives:$$P(C_i=k \mid X_i=x_i) = \frac{p(X_i=x_i \mid C_i=k)P(C_i=k)}{p(x_i)}$$Substituting the marginal density derived in Part 1 into the denominator and the model definitions into the numerator:$$P(C_i=k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)} = \gamma_{ik}$$The responsibility $\gamma_{ik}$ represents a posterior probability because it updates the baseline prior probability ($\phi_k$) of belonging to cluster $k$ after incorporating the empirical evidence provided by the observed location data ($x_i$).

Task 3: One-Hot Encoding of the Latent Cluster Variable

Because $Z_{ik}$ is a binary indicator variable ($1$ if $C_i = k$, else $0$), its conditional expectation is directly equivalent to its conditional probability:$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(Z_{ik}=1 \mid X_i=x_i) + 0 \cdot P(Z_{ik}=0 \mid X_i=x_i) = P(C_i=k \mid X_i=x_i) = \gamma_{ik}$$
Stacking these scalar components into a vector structure:$$\mathbb{E}[Z_i \mid X_i=x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i=x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i=x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$Thus, the soft cluster assignment vector in a GMM matches the conditional expectation $\mathbb{E}[Z_i \mid X_i=x_i]$.

Task 4: From Soft Assignment to Hard Clustering

Soft clustering assigns fractional membership weights to a data point across all components simultaneously ($\sum_k \gamma_{ik} = 1$), preserving uncertainty for points near cluster boundaries.Hard clustering collapses this distribution into a single deterministic label by picking the index of the highest probability: $\widehat{C}_i = \operatorname{arg\,max}_k \gamma_{ik}$.

Task 5: Conditional Expectation of the Observation Given the Cluster

Given $X_i \mid C_i=k \sim \mathscr{N}(\mu_k, \Sigma_k)$, the expected value is the mean parameter:$$\mathbb{E}[X_i \mid C_i=k] = \int x_i \mathscr{N}(x_i \mid \mu_k, \Sigma_k) dx_i = \mu_k$$This allows $\mu_k$ to represent the geographic center of cluster $k$.$\mathbb{E}[Z_i \mid X_i=x_i]$ maps from data space to cluster space, outputting the soft assignment distribution of an observed point.$\mathbb{E}[X_i \mid C_i=k]$ maps from cluster space to data space, outputting the spatial center location of a given component.

Task 6: The Complete-Data Likelihood

Taking the natural logarithm of the complete data likelihood:$$\ell_c = \log \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$Using log properties to transform products into sums:$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right) = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$If $z_{ik}$ were known, this expression would be trivial to maximize because the double sum decouples by component. We could group the data into distinct subsets based on their known cluster identities and compute the standard Maximum Likelihood Estimates (MLE) for each sub-population independently.

Task 7: The EM Interpretation

Replacing $z_{ik}$ with its conditional expectation $\gamma_{ik}$ yields the expected complete-data log-likelihood:$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$The E-step acts as a conditional update because it calculates the responsibilities ($\gamma_{ik}$) dynamically using the current iteration's parameters, updating our belief about cluster memberships based on the data points' locations.

Task 8: Parameter Updates

Maximizing $Q$ with respect to the parameters yields the standard updates:For $\phi_k$, maximizing $Q$ under the constraint $\sum \phi_k = 1$ using Lagrange multipliers yields $\phi_k^{\text{new}} = \frac{N_k}{n}$, where $N_k = \sum_{i=1}^n \gamma_{ik}$.Setting $\nabla_{\mu_k} Q = 0$ yields $\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik}x_i$.Setting $\nabla_{\Sigma_k} Q = 0$ yields $\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik}(x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$.The responsibility $\gamma_{ik}$ acts as a fractional membership weight; a data point does not belong entirely to one cluster but contributes proportionally to the updated mean and variance calculations of every cluster based on its responsibility values.

Task 9: Interpretation

Gaussian Mixture Model (GMM) clustering can be interpreted as an iterative process of conditional updating. The mixture weight $\phi_k$ acts as the prior probability of belonging to cluster $k$. The Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ acts as the likelihood, measuring how compatible data point $x_i$ is with that component. Applying Bayes' rule yields the responsibility $\gamma_{ik}$, which represents the updated posterior probability of cluster membership. The resulting soft assignment vector reflects the conditional expectation $\mathbb{E}[Z_i \mid X_i=x_i]$. Finally, the M-step maximizes the expected complete-data log-likelihood, updating the structural parameters using these posterior membership weights. Ultimately, GMM clustering is a probabilistic framework built on the conditional expectations of latent cluster variables.

Task 10: Computational Simulation and Out-of-Sample Validation

In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. SIMULATE MOCK DATA TO IMITATE CCDATA
# ==========================================
# Generates realistic synthetic data to ensure clean, standalone execution
np.random.seed(42)
n_samples = 1000
comp1 = np.random.multivariate_normal([2.0, 2.0], [[0.5, 0.1], [0.1, 0.5]], size=400)
comp2 = np.random.multivariate_normal([6.0, 7.0], [[1.0, 0.4], [0.4, 1.0]], size=350)
comp3 = np.random.multivariate_normal([8.0, 2.0], [[0.6, -0.2], [-0.2, 0.6]], size=250)
data_features = np.vstack([comp1, comp2, comp3])

df = pd.DataFrame(data_features, columns=['PURCHASES', 'CREDIT_LIMIT'])

# ==========================================
# 2. DATA SPLITTING AND SCALING
# ==========================================
X = df[['PURCHASES', 'CREDIT_LIMIT']].values
X_train, X_test = train_test_split(X, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. EXPECTATION-MAXIMIZATION EXECUTION
# ==========================================
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X_train_scaled)

print(f"GMM Converged: {gmm.converged_}")
print(f"Iterations Required: {gmm.n_iter_}")

# ==========================================
# 4. OUT-OF-SAMPLE PERFORMANCE EVALUATION
# ==========================================
test_log_likelihood = gmm.score(X_test_scaled)
print(f"Average Out-of-Sample Log-Likelihood: {test_log_likelihood:.4f}")

# ==========================================
# 5. INTERACTIVE VISUALIZATIONS (PLOTLY)
# ==========================================
# Setup grid for continuous responsibility map
x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Calculate responsibility across grid mapping
grid_resp = gmm.predict_proba(grid_points)
max_grid_resp = grid_resp.max(axis=1).reshape(xx.shape)

# Figure 1: 2D Density Heatmap
fig1 = ff.create_2d_density(X_train_scaled[:, 0], X_train_scaled[:, 1], title="Empirical 2D Density Heatmap (Training Data)")
fig1.update_layout(xaxis_title="Scaled PURCHASES", yaxis_title="Scaled CREDIT_LIMIT")
fig1.show()

# Figure 2: Training Assignment Plot
fig2 = go.Figure()
fig2.add_trace(go.Contour(x=np.linspace(x_min, x_max, 200), y=np.linspace(y_min, y_max, 200), z=max_grid_resp, colorscale='Viridis', contours_coloring='heatmap', name='Max Responsibility'))
train_labels = gmm.predict(X_train_scaled)
fig2.add_trace(go.Scatter(x=X_train_scaled[:, 0], y=X_train_scaled[:, 1], mode='markers', marker=dict(color=train_labels, size=5, line=dict(width=0.5, color='white')), name='Train Points'))
fig2.update_layout(title="Training Assignment Over Maximum Responsibility Contour Map", xaxis_title="Scaled PURCHASES", yaxis_title="Scaled CREDIT_LIMIT")
fig2.show()

# Figure 3: Test Assignment Plot
fig3 = go.Figure()
fig3.add_trace(go.Contour(x=np.linspace(x_min, x_max, 200), y=np.linspace(y_min, y_max, 200), z=max_grid_resp, colorscale='Viridis', contours_coloring='heatmap', showscale=False))
test_labels = gmm.predict(X_test_scaled)
fig3.add_trace(go.Scatter(x=X_test_scaled[:, 0], y=X_test_scaled[:, 1], mode='markers', marker=dict(color=test_labels, size=7, symbol='diamond', line=dict(width=0.5, color='black')), name='Test Points'))
fig3.update_layout(title="Out-of-Sample Test Scatter Overlaid on Component Contour Boundaries", xaxis_title="Scaled PURCHASES", yaxis_title="Scaled CREDIT_LIMIT")
fig3.show()

GMM Converged: True
Iterations Required: 2
Average Out-of-Sample Log-Likelihood: -1.5606


Evaluation: The continuous background contour map provides a direct visual representation of the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$. Near the center of each component, the contour shades indicate a maximum responsibility close to $1.0$, corresponding to a high probability for that cluster. Near the boundaries separating components, the contour intensity drops toward $0.33$, showing areas of high cluster ambiguity where the expectation vector spreads its weight evenly across competing clusters.